# Exercise 3: Topic Modeling 

We will walk through the basic pipeline to build and interpret a topic model, including understanding key evaluation metrics.  

Goal: Understanding the basics of topic modeling. 




In [1]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

## Outline: 

1. Step by step code for building a typical topic model pipeline using BERTopic. At
each step the pipeline the notebook will include information on the following: 
- The model being called.
- What does does and why it is included.
- An explanation of the underlying mechanics and parameters passed to the model.
- Links to documentation 

Run src/topic_model.py to train the model. 

```
uv run src/topic_model.py

```
Then return to this notebook again to evaluate the results using the information
stored in output/

2. Reviewing topics 
    - Mapping topic clusters back to the posts assigned to the cluster. 
    - What is returned in "-1"
    - How are topics labeled? 

2. Evaluating the model key metrics. 
    - intertopic distance
    - intratopic distance 
    - examining word probabilities 
    - Using c-TFI-DF to see which words are contributing the most to the topic cluster



## The BERTopic pipeline at a glance

BERTopic is a five-stage pipeline.  At each stage you can choose to use a
different model (i.e. K-Means not HDBSCAN for clustering)

```
  documents → pre-computed embeddings → UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERT/LLM
             (any model)  (reduce) (cluster) (vocabulary)    (re-weight)  (label)
```

1. **Embed** each document (we already did this in Notebook 2 — 384-d vectors).
2. **UMAP** projects the 384-d embeddings down to ~2–10 dimensions so that clustering is
   meaningful (HDBSCAN struggles in high dimensions).
3. **HDBSCAN** finds dense regions in the reduced space; everything else is labelled `-1`
   ("outlier").
4. **CountVectorizer + c-TF-IDF** treats each topic-cluster as one big "class document"
   and finds the n-grams that best distinguish it from the other classes.
5. **KeyBERT / LLM** turn those keywords into a human-readable label. (Referred
   to as "representation")

Reference: https://maartengr.github.io/BERTopic/algorithm/algorithm.html

## Step 0 — Load pre-computed embeddings

Re-embedding all posts during the workshop is slow. Instead we read the embeddings
produced by `src/preprocess.py` from `output/processed_posts.json`.

In [2]:
import json

import numpy as np

from src.config import OUTPUT
from src.data_models import PostDocument
from src.preprocess import extract_embedding_text

with open(OUTPUT / "processed_posts.json") as f:
    raw = json.load(f)

postdocs = [PostDocument(**postdoc) for postdoc in raw.values() if postdoc.get("doc_embedding")]
texts = [extract_embedding_text(postdoc) for postdoc in postdocs]
embeddings = np.array([postdoc.doc_embedding for postdoc in postdocs], dtype=np.float32)

print(f"Loaded {len(texts)} documents.")
print(f"Embeddings shape: {embeddings.shape}  (one row per doc, 384 dims)")

/Users/kas/dev/pycon_2026/topic-vector-search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 315 documents.
Embeddings shape: (315, 384)  (one row per doc, 384 dims)


## Step 1 — UMAP: dense, contextual → low-dimensional, separable

Why reduce? In 384-d space, distances between *all* points are large and similar — the
"curse of dimensionality". HDBSCAN works much better on a 2–10 dim projection that
preserves *local* structure.

Key parameters:
- `n_neighbors` — how much *local* vs *global* structure to preserve. Smaller = more
  local detail, more clusters.
- `min_dist` — how tightly packed neighbours can be in the projection. `0.0` lets
  clusters collapse to points (good for HDBSCAN).
- `metric="cosine"` — matches how our embeddings were trained.

Reference: https://umap-learn.readthedocs.io/en/latest/parameters.html

In [3]:
from umap import UMAP

# Note: Try n-neighbors=5 for more local structure, n_neighbors=50 for more global structure.
RANDOM_SEED = 99
umap_model = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED,
)
umap_embeddings = umap_model.fit_transform(embeddings)
print(f"UMAP-reduced embeddings shape: {umap_embeddings.shape}  (was {embeddings.shape})")
print(f"First 3 reduced points:\n{umap_embeddings[:3]}")

/Users/kas/dev/pycon_2026/topic-vector-search/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP-reduced embeddings shape: (315, 2)  (was (315, 384))
First 3 reduced points:
[[5.0012927 6.512189 ]
 [5.162998  6.825761 ]
 [7.46784   5.6262736]]


## Step 2 — HDBSCAN: density-based clustering

HDBSCAN finds regions in the UMAP space that are dense enough to be a cluster and
labels everything else as `-1` (outliers). Unlike k-means, you do **not** specify the
number of clusters — the data does.

Key parameters:
- `min_cluster_size` — the smallest number of points that can form a cluster. Larger →
  fewer, broader topics; smaller → more, more specific topics.
- `min_samples` — how conservative the algorithm is. Larger → more outliers.

Reference: https://hdbscan.readthedocs.io/en/latest/parameter_selection.html

In [4]:
from collections import Counter

from hdbscan import HDBSCAN

#Note: Try min_cluster_size=5 for more clusters, min_cluster_size=20 for fewer
#clusters. Also, min_samples controls how conservative the clustering is. Higher values
#will label more points as outliers (cluster label -1).

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=10, # defaults to the same value as min_cluster_size if not set
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)
cluster_labels = hdbscan_model.fit_predict(umap_embeddings)
counts = Counter(cluster_labels)
print("Cluster sizes (label -1 = outliers):")
for label, n in sorted(counts.items()):
    print(f"  topic {label:3d}: {n:4d} documents")

Cluster sizes (label -1 = outliers):
  topic  -1:   19 documents
  topic   0:   45 documents
  topic   1:   39 documents
  topic   2:   30 documents
  topic   3:   34 documents
  topic   4:   13 documents
  topic   5:   26 documents
  topic   6:   75 documents
  topic   7:   34 documents


## Step 3 — CountVectorizer: build the topic vocabulary

Once we have clusters, we still need *words* to describe them. BERTopic concatenates
every document in a cluster into one big "class document" and then runs a standard
`CountVectorizer` to build a vocabulary.

Key parameters (slide 30 — vocabulary pruning):
- `min_df` — drop terms that appear in fewer than this many documents (kills typos
  and one-offs).
- `max_df` — drop terms that appear in more than this fraction of documents
  (kills very common, near-stop-words).
- `ngram_range=(1, 3)` — keep unigrams, bigrams, and trigrams.
- `stop_words="english"` — drop the standard English stop-list.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

# These mirror the values currently used in src/topic_model.py:151.
# Note: Try min_df=2 to see how the vocabulary shrinks.
vectorizer = CountVectorizer(
    min_df=1,
    max_df=1.0,
    ngram_range=(1, 3),
    stop_words="english",
)
X = vectorizer.fit_transform(texts)
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Document-term matrix shape: {X.shape}")
print("Sample n-grams:", list(vectorizer.vocabulary_.keys())[:10])

Vocabulary size: 12,012
Document-term matrix shape: (315, 12012)
Sample n-grams: ['amid', 'chaos', 'city', 'finding', 'small', 'moments', 'soothe', 'like', 'gentle', 'cloud']


## Step 4 — c-TF-IDF: which words distinguish *this* topic from the others?

Plain TF-IDF compares a document against the rest of the corpus. **Class-based**
TF-IDF (c-TF-IDF) compares one *class* (topic) against the rest of the *classes*.
Result: the highest-scoring terms for a topic are the ones that are common *inside*
the topic but rare *outside* of it. That is the per-topic keyword ranking BERTopic
shows you.

Reference: https://maartengr.github.io/BERTopic/api/ctfidf.html

In [6]:
from bertopic.vectorizers import ClassTfidfTransformer

# Demo: build the per-cluster "class documents" and re-weight them with c-TF-IDF.
ctfidf = ClassTfidfTransformer()
# Group docs by HDBSCAN cluster label
from collections import defaultdict

cluster_docs: dict[int, list[str]] = defaultdict(list)
for label, text in zip(cluster_labels, texts, strict=True):
    cluster_docs[int(label)].append(text)

class_texts = [" ".join(cluster_docs[label]) for label in sorted(cluster_docs)]
X_class = vectorizer.transform(class_texts)
X_ctfidf = ctfidf.fit_transform(X_class).toarray()
print(f"c-TF-IDF matrix shape: {X_ctfidf.shape}  (one row per cluster)")
print("Sample c-TF-IDF values for first cluster:", X_ctfidf[0][:10])

c-TF-IDF matrix shape: (9, 12012)  (one row per cluster)
Sample c-TF-IDF values for first cluster: [0.00810533 0.00810533 0.00810533 0.         0.         0.
 0.         0.         0.         0.        ]


### Reading the c-TF-IDF output

Each float in `X_ctfidf` is a **c-TF-IDF score** for one (topic, term) pair. The matrix
has shape `(n_topics × vocab_size)`, so `X_ctfidf[i, j]` answers:

> *How much does term j distinguish topic i from all other topics?*

The score is computed as:

$$\text{c-TF-IDF}_{t,c} = \frac{f_{t,c}}{\sum_j f_{j,c}} \cdot \log\!\left(1 + \frac{A}{\sum_i f_{t,i}}\right)$$

Where:

- $f_{t,c}$ — frequency of term $t$ in the concatenated "class document" for cluster $c$
- $\sum_j f_{j,c}$ — total term count across that class document (normalizes for cluster size)
- $A$ — average number of words per class
- $\sum_i f_{t,i}$ — how many classes contain term $t$ at all (the IDF part)

In plain terms:

- **High positive score** → the term is frequent *within* this topic and rare *across other* topics — a strong topic keyword.
- **Near zero** → the term appears uniformly across topics; it doesn't distinguish anything.
- **Negative scores** are possible if the transformer subtracts a background frequency.

So the first 10 floats printed (`X_ctfidf[0][:10]`) are the c-TF-IDF weights for the
first 10 vocabulary terms in cluster 0. The terms with the *highest* values for each row
are what BERTopic surfaces as that topic's keywords.



**NOTE: BERTopic exposes the same c-TF-IDF scores through get_topic() — no need to
index into the raw matrix manually.**  

See below:**

`topic_model.get_topic(0)` 

## Step 5 — KeyBERTInspired: re-ranking keywords with embedding similarity

c-TF-IDF gives us statistically distinctive terms, but they can still be awkward
n-grams that are distinctive but not necesssarily representative of the topic. 
 **KeyBERTInspired** is a post-processing *representation model* that
re-ranks those candidates using embedding similarity, so the final keywords are
more semantically representative of the topic.

Reference: https://maartengr.github.io/BERTopic/api/representation/keybert.html

In [7]:
from bertopic.representation import KeyBERTInspired

keybert_model = KeyBERTInspired(
    top_n_words=10,  # The top n words to extract per topic.
    nr_repr_docs=5, # The number of representative documents to extract per cluster.
    nr_samples=500, # The number of candidate documents to extract per cluster.
    nr_candidate_words=100, # The number of candidate words per cluster.
    random_state=RANDOM_SEED,
)

## Step 6 — AI Labeling:  human-readable topic names

KeyBERTInspired gives us a ranked list of terms like `["rail transport", "high speed",
"california rail"]`. That is useful, but still jargon. The final stage passes those
keywords — plus a sample of representative documents — to an LLM which produces a
concise 3-word label such as **"California High-Speed Rail"**.

Note: Below we use Ollama, You can use any LLM backend to do this. The code is stored here: `src/ai_labeler.py` works


In [8]:
from bertopic.representation import OpenAI as BertTopicOpenAI
from src.config import OLLAMA_MODEL, OLLAMA_URL
import httpx
from openai import OpenAI
import logging

logger = logging.getLogger(__name__)

# Wrapped in try/fail in case OLLAMA is not running
def ai_labeler(prompt: str) -> BertTopicOpenAI | None:
    """
    Return an OpenAI-based labeler if possible, otherwise None.
    """
    try:
        resp = httpx.get(OLLAMA_URL.replace("/v1", "/api/tags"), timeout=2)
        if resp.status_code == 200:
            client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")
            return BertTopicOpenAI(
                client=client,
                model=OLLAMA_MODEL,
                exponential_backoff=True,
                chat=True,
                prompt=prompt,
                nr_docs=5,
            )
    except Exception:
        logger.warning("Could not connect to Ollama at %s. LLM-based labeling will be disabled.", OLLAMA_URL)
        return None

# BERTopic injects the topic's representative documents and keywords into the
# prompt, so we can just use placeholders here.
LABEL_PROMPT = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic
label of at most 3 words. Make sure it is in the following format:
topic: <topic label>
"""

llm_model = ai_labeler(LABEL_PROMPT)


## Step 7 — Putting it all together: the full BERTopic pipeline

Below is the same pipeline as `src/topic_model.py:train_topic_model`, condensed into
one cell. You don't need to run it now (it's slow), but read through the parameter
block — every parameter you tune is one of these.

In [9]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from src.config import EMBEDDING_MODEL_NAME

# Set prior to training, but called after the model is fit.
print("Load embedding model...", EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Ask BERTopic to use both KeyBERT and LLM-based labeling (if available) for topic representation.
representation_model = {
    "KeyBERT": keybert_model,
}
# LLM if backend available
if llm_model:
    representation_model["LLM"] = llm_model

topic_model = BERTopic(
    # BERTopic will re-embed tokens, documents as part of its internal pipeline, so we pass the same embedding model we used to generate the document embeddings.
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ClassTfidfTransformer(),
    representation_model=representation_model,
    min_topic_size=5,
    n_gram_range=(1, 3),
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)
topics, probabilities = topic_model.fit_transform(texts, embeddings)
print(f"Trained — found {topic_model.get_topic_info().shape[0] - 1} topics (plus -1 outlier).")

Load embedding model... all-MiniLM-L6-v2


2026-05-05 11:29:23,787 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-05 11:29:24,084 - BERTopic - Dimensionality - Completed ✓
2026-05-05 11:29:24,084 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-05 11:29:24,092 - BERTopic - Cluster - Completed ✓
2026-05-05 11:29:24,093 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 9/9 [00:54<00:00,  6.07s/it]
2026-05-05 11:30:19,556 - BERTopic - Representation - Completed ✓


Trained — found 8 topics (plus -1 outlier).


## Reviewing topics

Three core inspection methods:

- `get_topic_info()` — one row per topic with size and label.
- `get_topic(topic_id)` — top words + c-TF-IDF scores for one topic.
- `get_document_info(texts)` — joins each document back to its topic assignment.

Topic `-1` is the **outlier bucket** — documents HDBSCAN couldn't confidently place.
These are not bad documents, they're just not part of any dense cluster.

In [10]:
# Top-line summary of every topic
topic_model.get_topic_info().head(15)

,Topic,Count,Name,Representation,KeyBERT,LLM,Representative_Docs
0,-1,19,-1_just_kindness_small_journey,"[just, kindness, small, journey, food, group, ...","[kindness remind world, outward journey far, l...",[Reflections & Ventures],[the smell fades after the moment just passed ...
1,0,75,0_fog_moments_life_weather,"[fog, moments, life, weather, just, like, toda...","[sf weather, fog, heavy fog, foggy, weather, s...",[San Francisco Fog Chronicles],[karlthefog has been keeping san francisco in ...
2,1,45,1_music_vibes_musical_note_listen,"[music, vibes, musical_note, listen, musical_n...","[musical_notes vibes, musical_notes vibes need...",[Music and Melodies],[musical_note music for those who appreciate t...
3,2,39,2_water_open_open water_swimming,"[water, open, open water, swimming, cold, swim...","[open water swimmers, open water swimming, swi...",[Open Water Swimming Challenges],[open water swimming tests not only physical l...
4,3,34,3_space_rocket_spacex_astronauts,"[space, rocket, spacex, astronauts, new, stati...","[space tourism, space travel, nasa, space expl...",[Space Exploration and Travel],[human evolution species survival could depend...
5,4,34,4_home_creating_art_design,"[home, creating, art, design, terracotta, spac...","[herbhouse_with_garden diy projects, diy proje...",[Home Enhancement & Design],[creating an indoor garden can be as simple as...
6,5,30,5_high_rail_speed_high speed,"[high, rail, speed, high speed, speed rail, hi...","[californias high speed, california high speed...",[High Speed Rail California],[with its potential to revolutionize how calif...
7,6,26,6_cats_cat_kitty_feline,"[cats, cat, kitty, feline, purr, paw, instead,...","[purpose cats, cat paw, kitty, heart, cat paw ...",[Cat Healing and Joy],[in my recent experience i discovered sometime...
8,7,13,7_主人与小猫之间的爱是双向的你用心呵护它的同时也会回报以忠诚和快乐陪伴two_hearts...,[主人与小猫之间的爱是双向的你用心呵护它的同时也会回报以忠诚和快乐陪伴two_hearts ...,[kitty是人类最好的朋友之一不仅仅是因为它们的可爱外表还因为它们可以成为你忠实的伴侣给予...,[小猫与家庭],[cat_face kitty是人类最好的朋友之一不仅仅是因为它们的可爱外表还因为它们可以成...


In [11]:
# Top n-grams for topic 0 with their c-TF-IDF scores
topic_model.get_topic(0)

[('fog', 0.019399965624462338),
 ('moments', 0.013841434387306343),
 ('life', 0.013021276730295184),
 ('weather', 0.012146008006287035),
 ('just', 0.011688993735984582),
 ('like', 0.011398007280714002),
 ('today', 0.010910521461951478),
 ('san', 0.010548872031220654),
 ('daily', 0.010548872031220654),
 ('make', 0.010209335082118918)]

In [12]:
# How each document was labelled (truncate text for readability)
doc_info = topic_model.get_document_info(texts)
doc_info[["Document", "Topic", "Probability"]].head(10)

,Document,Topic,Probability
0,amid the chaos in my city finding small moment...,0,0.783426
1,this winter has hit hard but even heavier stor...,0,1.000000
2,crafted with a mixture of tea milk for balance...,0,0.286424
3,love the idea what hobbies did astronauts pick...,3,0.814776
4,did not feel the sun beat harsh that day yet f...,0,1.000000
5,here's a lighthearted challenge thoughif coffe...,0,0.300996
6,how often are your goals reached perfectly or ...,0,1.000000
7,every life feels at once a story film or paint...,0,0.566585
8,what unique rituals have people devised to sta...,0,0.284932
9,when artfully capturing life moments in histor...,0,0.588993


### Cross-reference with model artifacts

Once you run `uv run python -m src.topic_model`, the same information is dumped to
`output/`:

- `output/topic_assignments.csv` — `post_id`, `text`, `topic_id` for every doc.
- `output/topic_labels.json` — human-readable label per topic (LLM-generated when
  Ollama or OpenAI is available, falling back to the top-3 keywords).
- `output/topic_information.csv` — same as `get_topic_info()`.

These are what the demo app reads.

In [13]:
import pandas as pd

labels = json.loads((OUTPUT / "topic_labels.json").read_text())
assignments = pd.read_csv(OUTPUT / "topic_assignments.csv")
print("Topic labels (from disk):")
for tid, info in sorted(labels.items(), key=lambda kv: int(kv[0])):
    print(f"  {int(tid):3d}: {info['label']}")
print(f"\nAssignments dataframe ({len(assignments)} rows):")
assignments.head(10)

Topic labels (from disk):
    0: Home Design & Nature Integration
    1: Cat Parenting and Appreciation
    2: Fog and Weather in San Francisco
    3: Musical Discovery Vibes
    4: Open Water Swimming Safety
    5: Space Exploration & Tourism
    6: High-Speed Rail California Project

Assignments dataframe (154 rows):


,post_id,text,topic_id
0,1c9e469e-b3c6-4f2d-bc23-eea9edf596bf,paw_prints today's caturday is extra special b...,1
1,f5cad5e1-13c0-4214-8045-a61b302f00fc,cat_face kitty can be such a joy but they sure...,1
2,f00fe427-a70e-4d5d-b3b4-40c8dc284958,family_woman_girl_boy catmom and dad share equ...,1
3,df4fc396-abe4-4f5d-bc18-59d600f4932f,sparkles clawsome cats can sometimes be quite ...,1
4,6acbc397-607f-494b-9214-380f2920a42c,yarn fluffy kittens grow into beautiful feline...,1
5,e85cb5c5-4db5-4739-8c6f-3a707409ccce,自媒体大佬们注意了猫视频就是流量密码别再用狗来堆砌你的内容库了喵 paw_prints,1
6,0a4e465f-065b-4e4c-af57-26d737a44cae,cat_face kitty是人类最好的朋友之一不仅仅是因为它们的可爱外表还因为它们可以成为...,1
7,9d3f3316-5fd7-4414-938b-03f53a03e759,猫咪们的爪子不仅锋利能抓老鼠还能在家中发挥多才多艺的作用呢比如清理家具表面的小角落让家居环境...,1
8,9a90e9f9-a4a6-47f9-ba52-bb692a6dd6c8,catnapping is a real thing and it happens to g...,1
9,050beda7-38b3-4e67-ab24-1f8778cbb43d,猫咪们总是能找到自己最喜欢的小窝从高高的窗台上到柔软的毛毯上每个地方都有它们的理由house...,1


# Visualize the topic space

Two views worth knowing:

- **Intertopic distance map** (`visualize_topics`) — each circle is a topic, sized by
  the number of docs assigned. Topics close together in the map are semantically
  similar; widely-separated topics are well-distinguished. Outliers in the gaps
  between clusters are a clue that the model is finding *real* but loose structure.
- **Topic similarity heatmap** (`visualize_heatmap`) — pairwise cosine similarity
  between topic embeddings. Helpful when two topics are almost-but-not-quite the same.

Coherence vs diversity (slide 20):
- **Coherence** — do the words inside a topic actually go together? (high c-TF-IDF
  scores cluster.)
- **Diversity** — are the topics distinct from each other? (low off-diagonal in the
  heatmap.)

In [14]:
# Inline if running interactively. The same chart is also dumped to
# output/topic_visualization.html by src/topic_model.py.
topic_model.visualize_topics()

In [15]:
topic_model.visualize_heatmap()

## Run the full training script

```bash
uv run python -m src.topic_model
```

This writes every artifact above to `output/`. Reload the demo app, click into
the trending topics — you should see your freshly-trained labels and document
assignments.

# Exercise 

Time: 10 minutes

After running this notebook, rerun the code above, but this time with different
parameters passed to the model. 

1. Pair up with a partner. Each of you will change the parameters passed to the `src/topic_model.py` training
job. Tell your partner what you expect to happen. 
2. Retrain the model 
3. Review and evaluate the results of each persons changes together. Did the
   model run? Or did it break? If it broke, why did it break? How did the
   evaluation metrics change?
4. Report out your findings to the group (optional)